# Visualización de outliers y distribuciones:
El objetivo de este notebook es visualizar los outliers y distribuciones de todas las variables que vamos a utilizar.

---
## Importación de librerías

In [ ]:
# Módulo
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
from src.utils.files import read_file
from src.utils.config import popularity, load_env_file

load_env_file()

## Carga de datos
Configuración del Minio, poner en True para usar la información de Minio

In [ ]:
use_minio = False # Solo cambiar este parámetro
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df = read_file(popularity, minio)
df.head(2)

In [ ]:
df.columns

---
## Análisis

Tenemos muchas columnas con diferentes tipos de datos. Las numéricas y temporales serán fáciles de visualizar, mientras que las categóricas y textuales no tanto. Vamos ir una por una analizando cada variable.

### id 

No merece la pena visualizar el id al ser un número asignado con el único propósito de identificar los juegos, por esto no se considera ni que tenga una distribución ni valores atípicos.

### name

Los nombres no tienen una distribución como tal, lo máximo que podemos hacer es visualizar la longitud de los mismos. Se observa que la media está en torno a los 13 caracteres, teniendo algunos valores atípicos con más de 100 caracteres.

In [ ]:
df_names = df.copy()
df_names["len_name"] = df_names["name"].apply(len)
df_names = df_names[df_names["len_name"] < 150]
fig = px.histogram(df_names, x = df_names["len_name"], hover_name= df_names["name"], color_discrete_sequence  = ["lightseagreen"], title="<b>Distribución de longitudes de nombres</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

### short_description
No merece la pena visualizar la descripción de los juegos ya que no hay ninguna manera directa de ver distribuciones o valores atípicos.

### price_overview y price_range
La distribución de precios ya se muestra en el notebook dedicado a ello, además de experimentar con diferentes condiciones. Pero repasamos la distribución total de la variable. Destaca el rango de precios de 0.01€ a 4.99€ con más del 40% de los juegos. Además de que menos del 25% de los juegos cuesta más de 10€.

In [ ]:
def price_range(x):
    '''Dado el precio devuelve el rango en string.'''
    if x == 0:
        return 'Free'
    elif x > 0 and x < 5:
        return '[0.01,4.99]'
    elif x >= 5 and x < 10:
        return '[5.00,9.99]'
    elif x >= 10 and x < 15:
        return '[10.00,14.99]'
    elif x >= 15 and x < 20:
        return '[15.00,19.99]'
    elif x >= 20 and x < 30:
        return '[20.00,29.99]'
    elif x >= 30 and x < 40:
        return '[30.00,39.99]'
    elif x >= 40:
        return '>40'
    
df['price_range'] = df['price_overview'].apply(lambda x: price_range(x))
df['price_range']

In [ ]:
fig = go.Figure()
orden = [
    "Free", "[0.01,4.99]",
    "[5.00,9.99]", "[10.00,14.99]",
    "[15.00,19.99]", "[20.00,29.99]",
    "[30.00,39.99]", ">40"]

values_df = df["price_range"].value_counts()
values_df = values_df.reindex(orden)

fig.add_trace(go.Pie(values= values_df.values, labels= values_df.index, sort = False))
fig.update_layout(
    width = 800,
    title = {
        "text": "Proporción de los rangos de precio",
        "x": 0.5, "y":0.9, "font":dict(family="Verdana", size = 20, weight="bold")
    })

fig.show()

In [ ]:
fig = px.histogram(
    df,
    x = df["price_overview"],
    hover_name= df["name"],
    color_discrete_sequence  = ["lightseagreen"],
    title="<b>Distribución de precios</b>",)
fig.show()

Se observan ciertos datos atípicos de juegos que valen más de 100€ (se pueden considerar outliers pero también es algo influyente para el modelo de popularidad, un juego de más de 100€ probablemente no haga bien el mercado, en el modelo de precios se han limpiado juegos de más de 100€)

### num_languages
Esta es una variable que almacena el número de idiomas que soporta un juego.

In [ ]:
fig = px.histogram(df, x = df["num_languages"], hover_name= df["name"], color_discrete_sequence  = ["darkolivegreen"], title="<b>Distribución de número de lenguages soportados</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

La mayoría de los juegos sólo se encuentran disponibles en 1 o 2 idiomas como máximo mientras que tan sólo unos pocos son traducidos a una gran variedad de idiomas (normalmente estos son juegos exitosos que tienen los recursos para permitirse las traducciones). Por último destaca un grupo de juegos que indican tener todas las traducciones permitidas en Steam, probablemente esto no sea cierto y sea una estrategia para que dichos juegos lleguen a un mayor público.

Se observan también la presencia de 15 juegos que tienen 0 lenguajes, probablemente de desarrolladores que se les haya olvidado listarlos y unos 400 juegos que tienen más de 80 lenguajes, probablemente juegos que no tengan texto o hayan sido traducidos por máquinas.

### developers y publishers
Vamos a intentar hacer lo mismo que con los idomas, pero hay muchos más desarrolladores y publishers que idomas, por lo que visualizaremos solo los que tengan más de cierto número de juegos.

In [ ]:
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Distribución de Developers", "Distribución de Publishers"))

fig.add_trace(
    go.Histogram(
        x= df["num_juegos_previos_developers"],
        marker_color= "lightseagreen",
        xbins = dict(size = 3)
), row = 1, col = 1)

fig.add_trace(
    go.Histogram(
        x=df["num_juegos_previos_publishers"],
        marker_color= "steelblue",
        xbins = dict(size = 3)
),row = 1, col = 2)

fig.update_layout(height=800, width=1200, showlegend=False,)
fig.update_xaxes(title_text="Cantidad de juegos lanzados", range = [0, min(df["num_juegos_previos_developers"].max(), df["num_juegos_previos_publishers"].max())])
fig.update_yaxes(title_text="Número total", type = "log")
fig.show()

Las distribución tanto del número de juegos lanzados por publishers como developers es muy parecida, ambos muestran una distribución de cola larga con la excepción de que en este caso sí que existe un grupo de outliers relativamente grande conformado por developers/publishers que tienen una gran cantidad de juegos lanzados. Excluyendo a las grandes empresas de la industria de videojuegos, hemos comprobado que existen algunas compañías que lanzan cientos de juegos del mismo género y que son prácticamente idénticos entre sí.

In [ ]:
df['es_primer_juego_publishers'].value_counts()

In [ ]:
df['es_primer_juego_developers'].value_counts()

La mayoría de juegos presentes son el primer juego de un publisher o desarrolladora. Concuerda con el hecho de que la mayoría del catálogo de Steam sea por creadores pequeños que no suelen crear muchos juegos. Además de que los datos se correlacionan entre publisher y desarrolladora ya que normalmente son el mismo a no ser que sea una produccion más grande. 

In [ ]:
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Distribución de Developers", "Distribución de Publishers"))

fig.add_trace(
    go.Histogram(
        x= df["max_historico_reviews_developers"],
        marker_color= "lightseagreen",
        xbins = dict(size = 3)
), row = 1, col = 1)

fig.add_trace(
    go.Histogram(
        x=df["max_historico_reviews_publishers"],
        marker_color= "steelblue",
        xbins = dict(size = 3)
),row = 1, col = 2)

fig.update_layout(height=800, width=1200, showlegend=False,)
fig.update_xaxes(title_text="Cantidad de juegos lanzados", range = [0, min(df["max_historico_reviews_developers"].max(), df["max_historico_reviews_publishers"].max())])
fig.update_yaxes(title_text="Número máximo", type = "log")
fig.show()

Vemos que la mayoría de developers y publishers tienen un máximo histórico muy bajo, y que raramente hay proyectos que han sido muy exitosos.

In [ ]:
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Distribución de Developers", "Distribución de Publishers"))

fig.add_trace(
    go.Histogram(
        x= df["ema_reviews_developers"],
        marker_color= "lightseagreen",
        xbins = dict(size = 3)
), row = 1, col = 1)

fig.add_trace(
    go.Histogram(
        x=df["ema_reviews_publishers"],
        marker_color= "steelblue",
        xbins = dict(size = 3)
),row = 1, col = 2)

fig.update_layout(height=800, width=1200, showlegend=False,)
fig.update_xaxes(title_text="Cantidad de juegos lanzados", range = [0, min(df["ema_reviews_developers"].max(), df["ema_reviews_publishers"].max())])
fig.update_yaxes(title_text="Número máximo", type = "log")
fig.show()

### release_year
Ver la distribución de esta variable es sencillo

In [ ]:
df_release = df.copy()

df_release["release_year"] = df_release["release_year"].astype(int)

all_years = sorted(df["release_year"].unique())

fig = px.histogram(
    df_release,
    x="release_year",
    title="<b>Distribución de años de lanzamiento</b>",
    labels={'release_year': 'Año', 'count': 'Número de juegos'},
    color_discrete_sequence=["lightcoral"]
)

fig.update_layout(
    title={'x': 0.5, 'y': 0.9, 'xanchor': 'center', 'yanchor': 'top',
           'font': dict(family="Verdana", size=20)},
    bargap=0.1,
    plot_bgcolor="WhiteSmoke",
    xaxis=dict(
        tickmode="array",   
        tickvals=all_years,    
        ticktext=[str(y) for y in all_years]  
    )
)

fig.show()

### recomendaciones_totales
La distribución es una cola larga, teniendo casi todos los juegos muy pocas reseñas.

In [ ]:
limit = df["recomendaciones_totales"].quantile(0.98)
bins = np.linspace(0, limit, 25)
hist, _ = np.histogram(df["recomendaciones_totales"], bins=bins)
labels = [f"{int(bins[i])}-{int(bins[i+1])}" for i in range(len(bins)-1)]
hist_log = np.log1p(hist)


fig = go.Figure()

fig.add_trace(go.Bar(
    y=labels,
    x=hist_log,
    orientation="h",
    marker_color="seagreen",
    hovertemplate="Rango: %{y}<br>Juegos: %{customdata}",
    customdata=hist
))


tick_vals_raw = [0, 10, 100, 1000, 10000]
tick_vals_log = np.log1p(tick_vals_raw)

fig.update_layout(
    title={"text": "<b>Distribución de Recomendaciones Totales (Escala Log)</b>", "x": 0.5},
    xaxis=dict(
        tickvals=tick_vals_log,
        ticktext=[str(x) for x in tick_vals_raw],
        title="Número de juegos (Escala Log)"
    ),
    yaxis=dict(title="Rango de recomendaciones"),
    height=700
)

fig.show()

In [ ]:
print(f"Sólo el {round((df[df["recomendaciones_totales"] > 50].shape[0] / df.shape[0]) * 100,2)}% de los juegos supera las 50 reseñas")

### brillo
El brillo medio es un valor que va entre 0 y 255 (valores de negro y blanco), la media está entre 50 y 100 de brillo

In [ ]:
fig = px.histogram(df, x = df["brillo"], hover_name= df["name"], color_discrete_sequence  = ["crimson"], title="<b>Distribución del brillo de las cápsulas</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

### yt_score
El yt_score es una ponderación normalizada, por lo que está entre 0 y 1. Destaca que casi el 25% de los juegos tiene un score de 0, es decir, no tienen ningún video.

In [ ]:
fig = px.histogram(df, x = df["yt_score"], hover_name= df["name"], color_discrete_sequence  = ["crimson"], title="<b>Distribución yt_score</b>",)
fig.update_layout(plot_bgcolor = "WhiteSmoke", title={'x': 0.5,'y': 0.9,'xanchor': 'center','yanchor': 'top', 'font': dict(family="Verdana", size=20)})
fig.show()

### video_statistics

In [ ]:
metrics = ['viewCount', 'likeCount', 'favoriteCount', 'commentCount']
n_videos = 4

for metric in metrics:
    cols = [f'video_{i}_video_statistics.{metric}' for i in range(n_videos)]
    df[metric] = df[cols].sum(axis=1)

In [ ]:
df[metrics].describe()

favoriteCount es una columna de todo o casi todo 0s, ya que youtube eliminó el soporte para el conte de favoritos, devolviendo 0 para todos los valores.

In [ ]:
metricas = df[metrics].drop(columns=['favoriteCount']).apply(np.log1p)
cols = ['viewCount', 'likeCount', 'commentCount']
colors = ['#378ADD', '#1D9E75', '#D85A30']

fig = make_subplots(rows=1, cols=3, subplot_titles=cols)

for i, (col, color) in enumerate(zip(cols, colors), start=1):
    fig.add_trace(
        go.Histogram(x=metricas[col], nbinsx=50, name=col,
                     marker_color=color, opacity=0.85),
        row=1, col=i
    )

fig.update_yaxes(title_text='Número de vídeos', col=1)
fig.update_layout(
    title_text='Distribución de métricas',
    showlegend=False, bargap=0.04, template='plotly_white'
)
fig.show()

### géneros

In [ ]:
genre_cols = ['Action','Adventure', 'Casual', 'Early Access', 'Indie', 'RPG', 'Simulation', 'Strategy']

counts = df[genre_cols].sum().sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=counts.index,
    y=counts.values,
    marker_color='#378ADD',
    opacity=0.85
))

fig.update_layout(
    xaxis_title='Género',
    yaxis_title='Número de juegos',
    bargap=0.2
)

fig.show()

### categorías

In [ ]:
cat_cols = ['Custom Volume Controls',
       'Family Sharing', 'Full controller support', 'Multi-player',
       'Online Co-op', 'Online PvP', 'Partial Controller Support',
       'Playable without Timed Input', 'PvP', 'Remote Play Together',
       'Shared/Split Screen', 'Single-player', 'Steam Achievements',
       'Steam Cloud', 'Steam Leaderboards', 'Steam Trading Cards']

counts = df[cat_cols].sum().sort_values(ascending=False)

fig = go.Figure(go.Bar(
    x=counts.index,
    y=counts.values,
    marker_color='#1D9E75',
    opacity=0.85
))

fig.update_layout(
    xaxis_title='Categoría',
    yaxis_title='Número de juegos',
    bargap=0.2,
    xaxis_tickangle=-45
)

fig.show()